# Gerador de música com gramática generativa

Pipeline: **gramática harmônica** → progressão de acordes → **gramática melódica** por acorde → partes com instrumentos (melodia, harmonia, baixo, bateria) → arquivo **MIDI**.

Para trocar o estilo, altere `ESTILO` na seção de configuração e use **Kernel → Restart & Run All**.

## 1. Importações

In [ ]:
import random
import mido
from music21 import stream, meter, tempo, note, chord, instrument, converter

## 2. Configuração
- `ESTILO`: `'rock'` ou `'classica'`
- `TOM`: tônica da música (uma das notas de `NOTAS`, ex.: `'C'`, `'G'`, `'Eb'`)
- `SEMENTE`: fixe um número para repetir o mesmo resultado; use `None` para variar a cada execução

In [ ]:
ESTILO = 'rock'          # 'rock' ou 'classica'
TOM = 'C'
SEMENTE = None           # ex.: 42 para resultados reproduzíveis
TEMPOS_POR_ACORDE = 4.0  # um compasso 4/4 por acorde
ARQUIVO_SAIDA = 'music_generated.mid'

random.seed(SEMENTE)

## 3. Teoria básica: notas, intervalos e graus dos acordes
Cada grau (`'1'`, `'3'`, `'5'`…) é contado a partir da fundamental do acorde. Nos acordes maiores, `'b3'` é a *blue note* (terça menor), usada como cor na melodia.

In [ ]:
NOTAS = ['C','Db','D','Eb','E','F','Gb','G','Ab','A','Bb','B']

INTERVALOS = {
    'maior': {'1':0, '2':2, 'b3':3, '3':4, '5':7, '6':9},
    'menor': {'1':0, '2':2, '3':3, '5':7, '6':8},
}

def parse_acorde(nome):
    """'Am' -> ('A', 'menor'); 'C' -> ('C', 'maior')."""
    if nome.endswith('m'):
        return nome[:-1], 'menor'
    return nome, 'maior'

def graus_do_acorde(nome):
    """Mapa grau -> nome da nota para um acorde."""
    fund, qualidade = parse_acorde(nome)
    base = NOTAS.index(fund)
    return {g: NOTAS[(base + st) % 12] for g, st in INTERVALOS[qualidade].items()}

def grau_para_nota(grau, acorde):
    """Consulta o mapa (criado na seção 6). Levanta KeyError se o grau não existe no acorde."""
    return mapa_graus[acorde][grau]

def m21(nome, oitava):
    """Converte para a notação do music21: 'Eb' -> 'E-4'."""
    return nome[0] + nome[1:].replace('b', '-') + str(oitava)

## 4. Gramática melódica
Cada compasso tem duas metades de 2 tempos. Tempos **fortes** usam notas do acorde (1, 3, 5); tempos **fracos** usam notas de passagem (2, 6, b3).

In [ ]:
gramatica_melodia = {
    'MELODIA': [('COMPASSO', 1.0)],
    'COMPASSO': [('METADE METADE', 1.0)],              # 2 + 2 = 4 tempos, sempre
    'METADE': [
        ('FORTE 1.0 FRACA 1.0', 0.35),                 # semínima, semínima
        ('FORTE 1.0 FRACA 0.5 FRACA 0.5', 0.30),       # semínima, duas colcheias
        ('FORTE 0.5 FRACA 0.5 FORTE 1.0', 0.20),       # duas colcheias, semínima
        ('FORTE 2.0', 0.15),                           # mínima (respiro)
    ],
    'FORTE': [('1', 0.40), ('3', 0.35), ('5', 0.25)],
    'FRACA': [('2', 0.30), ('6', 0.25), ('5', 0.20), ('3', 0.15), ('b3', 0.10)],
}

def gerar(simbolo, regras):
    """Expande um símbolo de uma gramática probabilística até chegar nos terminais."""
    if simbolo not in regras:
        return [simbolo]
    opcoes, pesos = zip(*regras[simbolo])
    escolha = random.choices(opcoes, weights=pesos)[0]
    return [t for s in escolha.split() for t in gerar(s, regras)]

# Teste rápido: sequência de (grau, duração)
gerar('MELODIA', gramatica_melodia)

## 5. Gramáticas harmônicas
- **Clássica**: frase de pergunta (termina em V) e resposta (termina em I), seguindo a lógica Tônica → Subdominante → Dominante.
- **Rock**: blocos de verso/refrão com empréstimos modais (`bVII`, `bVI`).

Para criar regras novas, acrescente tuplas `(expansão, peso)` ou novos símbolos; os pesos não precisam somar 1.

In [ ]:
classica = {
    'PECA':        [('ANTECEDENTE CONSEQUENTE', 1.0)],
    'ANTECEDENTE': [('I S D', 0.5), ('I vi S D', 0.5)],     # termina em V (pergunta)
    'CONSEQUENTE': [('I S D I', 0.7), ('T S D I', 0.3)],    # termina em I (resposta)
    'T': [('I', 0.6), ('vi', 0.4)],
    'S': [('IV', 0.5), ('ii', 0.4), ('vi IV', 0.1)],
    'D': [('V', 1.0)],
}

rock = {
    'PECA':   [('VERSO VERSO', 0.4), ('VERSO REFRAO', 0.6)],
    'VERSO':  [('I D vi IV', 0.4), ('I bVII IV I', 0.3), ('I IV D IV', 0.3)],
    'REFRAO': [('vi IV I D', 0.5), ('IV I D vi', 0.3), ('bVI bVII I I', 0.2)],
    'D': [('V', 0.6), ('bVII', 0.4)],
}

GRAMATICAS_HARMONIA = {'rock': rock, 'classica': classica}

# Grau romano -> (semitons a partir do tom, sufixo do acorde)
GRAUS_ACORDE = {
    'I': (0, ''), 'ii': (2, 'm'), 'iii': (4, 'm'), 'IV': (5, ''),
    'V': (7, ''), 'vi': (9, 'm'),
    'bIII': (3, ''), 'bVI': (8, ''), 'bVII': (10, ''),
}

def grau_para_acorde(grau, tom='C'):
    st, sufixo = GRAUS_ACORDE[grau]
    return NOTAS[(NOTAS.index(tom) + st) % 12] + sufixo

## 6. Gerar a progressão de acordes

In [ ]:
graus = gerar('PECA', GRAMATICAS_HARMONIA[ESTILO])
progressao = [grau_para_acorde(g, tom=TOM) for g in graus]
mapa_graus = {ac: graus_do_acorde(ac) for ac in progressao}

print('Graus:     ', graus)
print('Progressão:', progressao)

## 7. Instrumentos (General MIDI)
Os números são programas do General MIDI começando em 0. A bateria usa o canal 10 do MIDI (índice 9).

In [ ]:
PROGRAMAS = {
    'rock':     {'melodia': 30, 'harmonia': 29, 'baixo': 33},  # guitarra distorcida, guitarra overdrive, baixo elétrico
    'classica': {'melodia': 40, 'harmonia': 48, 'baixo': 42},  # violino, naipe de cordas, violoncelo
}
CANAL_BATERIA = 9

def nova_parte(programa, nome, canal=None):
    p = stream.Part()
    p.partName = nome
    inst = instrument.Instrument()
    inst.midiProgram = programa
    inst.partName = nome
    if canal is not None:
        inst.midiChannel = canal
    p.insert(0, inst)
    return p

## 8. Montar as partes: melodia, harmonia, baixo e bateria

In [ ]:
melodia  = nova_parte(PROGRAMAS[ESTILO]['melodia'], 'Melodia')
harmonia = nova_parte(PROGRAMAS[ESTILO]['harmonia'], 'Harmonia')
baixo    = nova_parte(PROGRAMAS[ESTILO]['baixo'], 'Baixo')
melodia.insert(0, meter.TimeSignature('4/4'))
melodia.insert(0, tempo.MetronomeMark(number=120 if ESTILO == 'rock' else 90))

for acorde in progressao:
    # --- Melodia ---
    terminais = gerar('MELODIA', gramatica_melodia)
    usado = 0.0
    for grau, dur in zip(terminais[0::2], terminais[1::2]):
        dur = min(float(dur), TEMPOS_POR_ACORDE - usado)   # não passa do compasso
        if dur <= 0:
            break
        try:
            nome = grau_para_nota(grau, acorde)
        except KeyError:                                     # ex.: 'b3' em acorde menor
            nome = grau_para_nota('1', acorde)
        melodia.append(note.Note(m21(nome, 4), quarterLength=dur))
        usado += dur
    if usado < TEMPOS_POR_ACORDE:
        melodia.append(note.Rest(quarterLength=TEMPOS_POR_ACORDE - usado))

    # --- Harmonia: power chord no rock, tríade na clássica ---
    if ESTILO == 'rock':
        notas_h = [m21(grau_para_nota(g, acorde), 3) for g in ('1', '5')]
        notas_h.append(m21(grau_para_nota('1', acorde), 4))   # oitava em cima
        for _ in range(2):
            harmonia.append(chord.Chord(notas_h, quarterLength=2.0))
    else:
        notas_h = [m21(grau_para_nota(g, acorde), 3) for g in ('1', '3', '5')]
        harmonia.append(chord.Chord(notas_h, quarterLength=4.0))

    # --- Baixo: colcheias no rock, nota longa na clássica ---
    raiz = m21(grau_para_nota('1', acorde), 2)
    if ESTILO == 'rock':
        for _ in range(8):
            baixo.append(note.Note(raiz, quarterLength=0.5))
    else:
        baixo.append(note.Note(raiz, quarterLength=4.0))

partes = [melodia, harmonia, baixo]

# --- Bateria (só no rock) ---
if ESTILO == 'rock':
    bateria = nova_parte(0, 'Bateria', canal=CANAL_BATERIA)
    BUMBO, CAIXA, CHIMBAL = 36, 38, 42
    for _ in progressao:
        for i in range(8):                      # 8 colcheias por compasso
            golpes = [CHIMBAL]
            if i in (0, 4):
                golpes.append(BUMBO)            # tempos 1 e 3
            if i in (2, 6):
                golpes.append(CAIXA)            # tempos 2 e 4
            bateria.append(chord.Chord([note.Note(midi=g) for g in golpes], quarterLength=0.5))
    partes.append(bateria)

partitura = stream.Score(partes)

## 9. Salvar o MIDI
Depois de gravar, o arquivo passa por uma correção com `mido`: garante a troca de instrumento em cada trilha e coloca a bateria no canal 10, independentemente de como a versão do music21 distribuiu os canais.

In [ ]:
partitura.write('midi', fp=ARQUIVO_SAIDA)

def corrigir_canais(caminho, programas, tem_bateria):
    """Força um canal e um program_change por trilha; a última trilha vira bateria (canal 10)."""
    mid = mido.MidiFile(caminho)
    trilhas_notas = [t for t in mid.tracks if any(m.type == 'note_on' for m in t)]
    for idx, trilha in enumerate(trilhas_notas):
        eh_bateria = tem_bateria and idx == len(trilhas_notas) - 1
        canal = CANAL_BATERIA if eh_bateria else idx
        programa = 0 if eh_bateria else programas[idx]
        novas = [m for m in trilha if m.type != 'program_change']
        for m in novas:
            if hasattr(m, 'channel'):
                m.channel = canal
        insere = next((i for i, m in enumerate(novas) if not m.is_meta), len(novas))
        novas.insert(insere, mido.Message('program_change', channel=canal, program=programa, time=0))
        trilha[:] = novas
    mid.save(caminho)

ordem = [PROGRAMAS[ESTILO][k] for k in ('melodia', 'harmonia', 'baixo')]
corrigir_canais(ARQUIVO_SAIDA, ordem, tem_bateria=(ESTILO == 'rock'))
print('MIDI salvo em', ARQUIVO_SAIDA)

## 10. Conferir os instrumentos gravados no arquivo

In [ ]:
for i, trilha in enumerate(mido.MidiFile(ARQUIVO_SAIDA).tracks):
    programas = [(m.channel + 1, m.program) for m in trilha if m.type == 'program_change']
    notas = sum(1 for m in trilha if m.type == 'note_on' and m.velocity > 0)
    print(f'Trilha {i}: (canal, programa) = {programas}, notas = {notas}')

## 11. Ouvir no notebook
O player do notebook roda no navegador e às vezes não carrega todos os timbres. Se soar tudo como piano, abra o arquivo `.mid` no MuseScore, VLC ou numa DAW (Reaper, LMMS, GarageBand).

In [ ]:
converter.parse(ARQUIVO_SAIDA).show('midi')

## 12. (Opcional) Ouvir um MIDI de referência

In [ ]:
musica_ref = converter.parse('./samples/Goodbye Yellow Brick Road Elton John.mid')
musica_ref.show('midi')